# Byte Pair Encoding (BPE) Tokenizer

This notebook explains the **Byte Pair Encoding (BPE)** tokenization algorithm step by step. BPE was first used as a text compression idea and later became a popular subword tokenization method for Transformer models such as GPT, GPT-2, RoBERTa, BART, and DeBERTa.

The goal of this lecture is to make the algorithm visible. We will train a tiny BPE tokenizer from scratch, inspect every merge it learns, and then use the learned merge rules to tokenize new words.

Reference: Hugging Face LLM Course, Chapter 6, section 5, "Byte-Pair Encoding tokenization": https://huggingface.co/learn/llm-course/chapter6/5

## Learning Objectives

By the end of this notebook, you should be able to:

- Explain why language models use **subword tokens** instead of only words or characters.
- Describe the BPE training algorithm: start with characters, count adjacent pairs, merge the most frequent pair, and repeat.
- Apply learned BPE merge rules to tokenize a new word or sentence.
- Interpret vocabulary growth and understand why frequent pieces become single tokens.
- Distinguish basic character-level BPE from **byte-level BPE** used by GPT-style tokenizers.

## 1. Why Tokenization Matters

A neural language model does not directly read a Python string such as:

```text
"tokenization is useful"
```

It reads a sequence of integer token IDs:

```text
[9152, 374, 5505]
```

A **tokenizer** is the component that converts text into tokens and token IDs. The model then looks up each token ID in an embedding table.

There are three simple ways to tokenize text:

| Approach | Example for `playing` | Strength | Weakness |
| --- | --- | --- | --- |
| Word-level | `playing` | Easy to interpret | Huge vocabulary; unknown words are common |
| Character-level | `p l a y i n g` | Can represent any word | Long sequences; harder for the model to learn larger units |
| Subword-level | `play ing` or `play ##ing` | Balanced vocabulary and sequence length | Tokens are less intuitive than words |

BPE is a **subword tokenization** algorithm. It learns common pieces from data instead of relying on hand-written rules.

## 2. The Main Idea of BPE

BPE starts with very small tokens, usually characters or bytes.

Then it repeatedly asks:

> Which adjacent pair of tokens appears most often in the training corpus?

That pair is merged into a new token. The new token is added to the vocabulary. After many merges, frequent character sequences become single tokens.

For example, if `u` followed by `g` appears very often, BPE may learn this merge:

```text
("u", "g") -> "ug"
```

After this rule is learned, the word `hug` can be represented as:

```text
["h", "ug"]
```

Later, if `h` followed by `ug` is frequent, BPE may learn:

```text
("h", "ug") -> "hug"
```

Now `hug` can be represented as one token:

```text
["hug"]
```

## 3. BPE Training Algorithm

A BPE tokenizer is usually trained after two earlier steps:

1. **Normalization**: optional text cleanup, such as lowercasing or Unicode normalization.
2. **Pre-tokenization**: splitting text into rough units, often words and punctuation.

### Where pre-tokenization fits

The full path from text to model input is:

```text
raw text → normalization → pre-tokenization → BPE merges inside each chunk → token IDs
```

**Pre-tokenization** splits raw text into rough, word-like chunks *before* BPE runs. BPE then learns and applies merges **only inside a chunk, never across chunk boundaries**. This step matters for three reasons:

1. **It controls what can merge.** Without chunk boundaries, BPE would count pairs across the entire text. It could then learn tokens that span two words, such as `"e th"` from `"the theory"`, and frequent word pairs would become tokens.
2. **It makes training tractable.** After pre-tokenization, training only needs each unique chunk and how often it occurs. If `"This"` appears 3 times, it is processed once with weight 3. This is why the worked example in Section 4 starts from a dictionary of words and frequencies (`word_freqs`), not from the raw text.
3. **It decides how spaces and punctuation are treated.** Should a space become its own token or attach to the next word? Can `,` ever merge with a letter? The pre-tokenizer makes these choices; BPE does not.

For example, GPT-2's pre-tokenizer, a regular expression with separate rules for letters, numbers, punctuation, contractions, and spaces, splits a sentence like this:

```text
"Hello, I'm learning tokenization in 2024!"
→ ['Hello', ',', ' I', "'m", ' learning', ' tokenization', ' in', ' 2024', '!']
```

Notice three design choices:

- **Spaces stay attached to the following word** (`' learning'`), so no information about spacing is lost.
- **Contractions are split off** (`"'m"`).
- **Letters, digits, and punctuation are never in the same chunk**, so BPE cannot learn a token such as `"n!"`.

Other tokenizers make different choices. BERT's WordPiece pre-tokenizer splits on whitespace and punctuation and discards the spaces. SentencePiece, used by T5, skips this step and treats the space as an ordinary symbol `▁` in the raw text. Section 8 builds a simple pre-tokenizer of our own.

### The training loop

For a classroom version of BPE, the training loop is:

1. Count how often each word appears in the corpus.
2. Create the initial vocabulary from all characters in those words.
3. Split each word into characters.
4. Count all adjacent token pairs, weighted by word frequency.
5. Merge the most frequent pair everywhere it appears.
6. Add the merged pair to the vocabulary.
7. Repeat until the vocabulary reaches the desired size or the desired number of merges is learned.

The important point: **BPE is greedy**. At each step, it chooses the best pair right now. It does not search for the globally best vocabulary.

## 4. A Tiny Worked Example

We will use the same style of toy vocabulary used in the Hugging Face lesson:

```text
("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)
```

The numbers are word frequencies. For example, `hug` appears 10 times and `pun` appears 12 times.

The initial tokens are characters:

```text
hug  -> h u g
pug  -> p u g
pun  -> p u n
bun  -> b u n
hugs -> h u g s
```

The initial vocabulary is:

```text
b, g, h, n, p, s, u
```

In [1]:
from collections import Counter, defaultdict

word_freqs = {
    "hug": 10,
    "pug": 5,
    "pun": 12,
    "bun": 4,
    "hugs": 5,
}

splits = {word: list(word) for word in word_freqs}
base_vocab = sorted({char for word in word_freqs for char in word})

print("Initial vocabulary:", base_vocab)
print("Initial splits:")
for word, split in splits.items():
    print(f"  {word:4s} -> {split}  frequency={word_freqs[word]}")

Initial vocabulary: ['b', 'g', 'h', 'n', 'p', 's', 'u']
Initial splits:
  hug  -> ['h', 'u', 'g']  frequency=10
  pug  -> ['p', 'u', 'g']  frequency=5
  pun  -> ['p', 'u', 'n']  frequency=12
  bun  -> ['b', 'u', 'n']  frequency=4
  hugs -> ['h', 'u', 'g', 's']  frequency=5


### Counting Adjacent Pairs

BPE counts pairs inside each split word. The count is weighted by word frequency.

For `hug` with frequency 10, the pairs are:

```text
(h, u) and (u, g)
```

So both pairs receive `+10`.

For `hugs` with frequency 5, the pairs are:

```text
(h, u), (u, g), and (g, s)
```

So each receives `+5`.

In [2]:
def compute_pair_freqs(splits, word_freqs):
    """Count adjacent token pairs across all split words."""
    pair_freqs = defaultdict(int)

    for word, freq in word_freqs.items():
        split = splits[word]
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq

    return pair_freqs

pair_freqs = compute_pair_freqs(splits, word_freqs)

print("Pair frequencies before the first merge:")
# pair_freqs.items() gives ((token_1, token_2), frequency) entries.
# The sorting key is (-frequency, pair):
#   - negative frequency sorts larger counts first, because sorted() is ascending by default.
#   - pair sorts alphabetically when two pairs have the same frequency.
for pair, freq in sorted(pair_freqs.items(), key=lambda item: (-item[1], item[0])):
    print(f"  {pair}: {freq}")

Pair frequencies before the first merge:
  ('u', 'g'): 20
  ('p', 'u'): 17
  ('u', 'n'): 16
  ('h', 'u'): 15
  ('g', 's'): 5
  ('b', 'u'): 4


The most frequent pair is merged first.

In this example, `("u", "g")` appears inside `hug`, `pug`, and `hugs`, for a total frequency of 20. So the first learned merge is:

```text
("u", "g") -> "ug"
```

In [3]:
def merge_pair(pair, splits):
    """Merge one pair everywhere it appears in every split word."""
    a, b = pair
    new_splits = {}

    for word, split in splits.items():
        merged = []
        i = 0
        while i < len(split):
            if i < len(split) - 1 and split[i] == a and split[i + 1] == b:
                merged.append(a + b)
                i += 2
            else:
                merged.append(split[i])
                i += 1
        new_splits[word] = merged

    return new_splits

best_pair = max(pair_freqs, key=pair_freqs.get)
print("Best pair:", best_pair, "frequency=", pair_freqs[best_pair])

splits_after_first_merge = merge_pair(best_pair, splits)

print("Splits after merging", best_pair, "->", "".join(best_pair))
for word, split in splits_after_first_merge.items():
    print(f"  {word:4s} -> {split}")

Best pair: ('u', 'g') frequency= 20
Splits after merging ('u', 'g') -> ug
  hug  -> ['h', 'ug']
  pug  -> ['p', 'ug']
  pun  -> ['p', 'u', 'n']
  bun  -> ['b', 'u', 'n']
  hugs -> ['h', 'ug', 's']


### Visual walkthrough: count → merge → recount

Run the next cell after the pair-counting and merging functions above. Click **Next merge** to follow five training steps; **Reset** returns to character tokens. Each view shows the current splits, their frequency-weighted pair counts, and the result of merging the winning pair everywhere it occurs. Highlighted tiles mark the selected pair before merging and the new token afterward.

The bar lengths represent **pair occurrences weighted by word frequency**, not the number of distinct words. For the first step, `count(u, g) = 10 + 5 + 5 = 20`. After a merge, we recount pairs using the updated tokens: a candidate can now include a previously merged token such as `h + ug`.

The vocabulary retains its existing tokens and adds one new token per merge, even though the displayed word splits become shorter. The ordered rule list also shows how an unseen word, `bug`, can reuse learned pieces without counting pairs again.

This uses IPython's HTML display and optionally `ipywidgets` for buttons. If `ipywidgets` is unavailable, it displays all five steps as static panels. If buttons are not supported by your notebook viewer, set `BPE_INTERACTIVE = False` and rerun the cell. No downloads or training of a language model are needed.


In [13]:
from html import escape
from IPython.display import HTML, display

BPE_INTERACTIVE = True  # Set False for static panels or notebook export.
BPE_VISUAL_MERGES = 5

# Reuse the notebook's algorithm, keeping the visual's state separate from splits.
visual_splits = {word: list(word) for word in word_freqs}
visual_base_vocab = sorted({char for word in word_freqs for char in word})
visual_steps = []
visual_rules = []

for _ in range(BPE_VISUAL_MERGES):
    counts = compute_pair_freqs(visual_splits, word_freqs)
    if not counts:
        break
    # Match train_bpe: ties use the first pair encountered in corpus order.
    winner = max(counts, key=counts.get)
    after = merge_pair(winner, visual_splits)
    visual_rules.append((winner, "".join(winner), counts[winner]))
    visual_steps.append({
        "before": visual_splits,
        "counts": dict(counts),
        "winner": winner,
        "after": after,
        "rules": list(visual_rules),
    })
    visual_splits = after


def bpe_tiles(tokens, pair=None, merged_token=None):
    """Draw token boundaries and highlight the selected merge."""
    selected = set()
    if pair is not None:
        i = 0
        while i < len(tokens) - 1:
            if tuple(tokens[i:i + 2]) == pair:
                selected.update((i, i + 1))
                i += 2
            else:
                i += 1
    tiles = []
    for i, token in enumerate(tokens):
        active = i in selected or token == merged_token
        color = "#fef3c7" if active else "#f1f5f9"
        border = "#a16207" if active else "#64748b"
        tiles.append(
            f'<span style="display:inline-block;margin:3px;padding:5px 9px;'
            f'border:2px solid {border};border-radius:6px;background:{color};'
            f'color:#172033;font-family:monospace">{escape(token)}</span>'
        )
    return "".join(tiles)


def bpe_word_rows(splits_to_show, pair=None, merged_token=None):
    return "".join(
        f'<div style="margin:7px 0"><span style="display:inline-block;'
        f'width:85px;font-family:monospace">{escape(word)} ×{word_freqs[word]}</span>'
        f'{bpe_tiles(tokens, pair, merged_token)}</div>'
        for word, tokens in splits_to_show.items()
    )


def bpe_visual_html(step_index):
    """Render one merge and the rules learned up to that step."""
    step = visual_steps[step_index]
    winner = step["winner"]
    new_token = "".join(winner)
    counts = step["counts"]
    bars = []
    # Stable sorting keeps tied pairs in the same order as the algorithm.
    for pair, count in sorted(counts.items(), key=lambda item: -item[1]):
        active = pair == winner
        width = 100 * count / max(counts.values())
        color = "#a16207" if active else "#64748b"
        label = escape(f"{pair[0]} + {pair[1]}")
        bars.append(
            f'<div style="margin:8px 0"><div style="font-family:monospace">'
            f'{label}: <b>{count}</b>{" ← selected" if active else ""}</div>'
            f'<div style="background:#e2e8f0;height:10px;border-radius:4px">'
            f'<div style="width:{width}%;height:10px;background:{color};'
            f'border-radius:4px"></div></div></div>'
        )
    rules_html = "".join(
        f'<li><code>{escape(a)} + {escape(b)} → {escape(token)}</code>'
        f' (training count: {count})</li>'
        for (a, b), token, count in step["rules"]
    )
    # Apply the recorded rules to an unseen word; inference does not recount pairs.
    unseen = list("bug")
    for pair, _, _ in step["rules"]:
        unseen = merge_pair(pair, {"bug": unseen})["bug"]
    parts = [
        ("① Current token splits", bpe_word_rows(step["before"], pair=winner)),
        ("② Weighted adjacent-pair counts", "".join(bars)),
        ("③ After merging everywhere", bpe_word_rows(step["after"], merged_token=new_token)),
    ]
    panels = "".join(
        f'<section style="flex:1;min-width:240px;padding:12px;border:1px solid '
        f'#cbd5e1;border-radius:8px"><h4 style="margin:0 0 12px">{title}</h4>'
        f'{content}</section>' for title, content in parts
    )
    return (
        '<div style="background:white;color:#172033;padding:16px;'
        'border:1px solid #cbd5e1;border-radius:10px;font-family:sans-serif">'
        f'<h3 style="margin-top:0">BPE training — merge {step_index + 1}</h3>'
        f'<div style="display:flex;flex-wrap:wrap;gap:12px">{panels}</div>'
        '<p><b>↻ Recount pairs in the updated splits → select the next winner.</b></p>'
        f'<p>Vocabulary: {len(visual_base_vocab)} base tokens + '
        f'{len(step["rules"])} learned tokens = '
        f'<b>{len(visual_base_vocab) + len(step["rules"])}</b> tokens. '
        'Existing vocabulary entries remain available.</p>'
        f'<b>Ordered learned rules</b><ol>{rules_html}</ol>'
        f'<p><b>Tokenization of unseen word bug:</b> '
        f'{bpe_tiles(list("bug"))} → {bpe_tiles(unseen)}</p>'
        '<p>Training counts pairs and learns rules. Tokenization applies those rules '
        'to new text without counting pairs.</p></div>'
    )


try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if BPE_INTERACTIVE and widgets is not None:
    visual_position = {"index": 0}
    next_merge_button = widgets.Button(description="Next merge", button_style="primary")
    reset_merge_button = widgets.Button(description="Reset")
    visual_panel = widgets.HTML()

    def show_bpe_step():
        visual_panel.value = bpe_visual_html(visual_position["index"])
        next_merge_button.disabled = visual_position["index"] == len(visual_steps) - 1

    def advance_bpe_step(_):
        visual_position["index"] = min(visual_position["index"] + 1, len(visual_steps) - 1)
        show_bpe_step()

    def reset_bpe_step(_):
        visual_position["index"] = 0
        show_bpe_step()

    next_merge_button.on_click(advance_bpe_step)
    reset_merge_button.on_click(reset_bpe_step)
    show_bpe_step()
    display(widgets.VBox([widgets.HBox([next_merge_button, reset_merge_button]), visual_panel]))
else:
    for step_index in range(len(visual_steps)):
        display(HTML(bpe_visual_html(step_index)))


## 5. Training Several BPE Merges

Now we can put the steps into a training function.

The function below returns:

- `vocab`: the initial character vocabulary plus every merged token.
- `merges`: the ordered list of merge rules.
- `splits`: the final split form of every training word.

The order of merge rules matters. During tokenization, we apply the rules in the same order they were learned.

In [5]:
def train_bpe(word_freqs, num_merges):
    # Start with the smallest possible tokens: characters.
    # Example: "hug" becomes ["h", "u", "g"].
    # During training, these lists will gradually contain larger subword tokens.
    splits = {word: list(word) for word in word_freqs}

    # The initial vocabulary contains each unique character found in the training words.
    # sorted(...) makes the printed vocabulary deterministic and easy to read.
    vocab = sorted({char for word in word_freqs for char in word})

    # This will store the learned BPE rules in order.
    # Each entry will look like: (("u", "g"), "ug", 20)
    # meaning: merge ("u", "g") into "ug", and its training frequency was 20.
    merges = []

    # Learn one merge rule per iteration.
    for step in range(num_merges):
        # Count all adjacent token pairs in the current splits.
        # These counts are weighted by the word frequencies from word_freqs.
        pair_freqs = compute_pair_freqs(splits, word_freqs)

        # If there are no adjacent pairs left, there is nothing more to merge.
        # This can happen if every training word has already become a single token.
        if not pair_freqs:
            break

        # Choose the pair with the highest frequency.
        # This is the greedy step in BPE: the best pair right now is merged next.
        best_pair = max(pair_freqs, key=pair_freqs.get)

        # Create the new token by joining the two tokens in the best pair.
        # Example: ("u", "g") becomes "ug".
        merged_token = "".join(best_pair)

        # Apply the new merge rule to every word split.
        # Wherever best_pair appears as adjacent tokens, it is replaced by merged_token.
        splits = merge_pair(best_pair, splits)

        # Save the merge rule so we can later tokenize new text using the same order.
        # The order matters: BPE tokenization applies merges in the order learned.
        merges.append((best_pair, merged_token, pair_freqs[best_pair]))

        # Add the newly created token to the vocabulary.
        vocab.append(merged_token)

        # Print a trace of what the algorithm learned at this step.
        print(f"Step {step + 1}: merge {best_pair} -> {merged_token!r}  frequency={pair_freqs[best_pair]}")

    # Return the final vocabulary, learned merge rules, and final training-word splits.
    return vocab, merges, splits

vocab, merges, final_splits = train_bpe(word_freqs, num_merges=5)

print("\nFinal vocabulary:")
print(vocab)

print("\nFinal training-word splits:")
for word, split in final_splits.items():
    print(f"  {word:4s} -> {split}")

Step 1: merge ('u', 'g') -> 'ug'  frequency=20
Step 2: merge ('u', 'n') -> 'un'  frequency=16
Step 3: merge ('h', 'ug') -> 'hug'  frequency=15
Step 4: merge ('p', 'un') -> 'pun'  frequency=12
Step 5: merge ('p', 'ug') -> 'pug'  frequency=5

Final vocabulary:
['b', 'g', 'h', 'n', 'p', 's', 'u', 'ug', 'un', 'hug', 'pun', 'pug']

Final training-word splits:
  hug  -> ['hug']
  pug  -> ['pug']
  pun  -> ['pun']
  bun  -> ['b', 'un']
  hugs -> ['hug', 's']


### What Did BPE Learn?

The tokenizer learned frequent chunks from the training words.

A frequent pair such as `u` + `g` became `ug`. Then a later merge could use that newly created token, such as `h` + `ug` becoming `hug`.

This is how BPE grows from characters into longer subwords:

```text
characters -> short subwords -> longer subwords -> sometimes whole words
```

BPE does not know grammar. It only knows frequency patterns in the training corpus.

## 6. BPE Tokenization Algorithm

After training, tokenization uses the learned merge rules.

For a new word:

1. Split the word into characters.
2. Apply the first learned merge wherever possible.
3. Apply the second learned merge wherever possible.
4. Continue through all learned merges in order.
5. Return the final list of tokens.

The tokenizer does not count pairs during tokenization. Pair counting happens only during training.

In [6]:
def tokenize_word(word, merges, vocab=None, unk_token="[UNK]"):
    """Tokenize one word by applying learned BPE merges in order."""
    # Start from the smallest units, just like BPE training did.
    # Example: "hug" becomes ["h", "u", "g"].
    split = list(word)

    # If a base vocabulary is provided, check whether each starting character is known.
    # Unknown characters are replaced with unk_token.
    # Example: if "m" was never seen during training, "mug" starts as ["[UNK]", "u", "g"].
    if vocab is not None:
        known_base_symbols = set(vocab)
        split = [char if char in known_base_symbols else unk_token for char in split]

    # Apply the learned merge rules in the same order they were learned during training.
    # The order matters because later merges may depend on tokens created by earlier merges.
    for pair, merged_token, _freq in merges:
        # new_split will store the result after applying this one merge rule.
        new_split = []

        # i is the current position in the existing split.
        # We scan from left to right looking for the current pair.
        i = 0
        while i < len(split):
            # Check whether the current token and the next token match the learned pair.
            # The first condition makes sure split[i + 1] exists before we read it.
            if i < len(split) - 1 and split[i] == pair[0] and split[i + 1] == pair[1]:
                # If the pair matches, replace the two tokens with the merged token.
                # Example: ["u", "g"] becomes ["ug"].
                new_split.append(merged_token)

                # Move ahead by 2 because both tokens in the pair have been consumed.
                i += 2
            else:
                # If there is no match at this position, keep the current token unchanged.
                new_split.append(split[i])

                # Move ahead by 1 because only the current token was consumed.
                i += 1

        # The output of this merge becomes the input to the next merge rule.
        split = new_split

    # After all learned merges have been applied, split is the final BPE tokenization.
    return split

base_vocab = sorted({char for word in word_freqs for char in word})

for word in ["hug", "bug", "pug", "unhug", "mug", "thug"]:
    print(f"{word:6s} -> {tokenize_word(word, merges, vocab=base_vocab)}")

hug    -> ['hug']
bug    -> ['b', 'ug']
pug    -> ['pug']
unhug  -> ['un', 'hug']
mug    -> ['[UNK]', 'ug']
thug   -> ['[UNK]', 'hug']


### Unknown Characters

Our toy tokenizer has a small base vocabulary:

```text
b, g, h, n, p, s, u
```

So if we ask it to tokenize `mug`, the character `m` is unknown. This is why the output contains `[UNK]`.

Real GPT-2-style tokenizers avoid this problem with **byte-level BPE**. Instead of starting from Unicode characters, they start from bytes. Since there are only 256 possible byte values, any text can be represented without needing an unknown character token.

## 7. How GPT-2 Converts Text to Bytes Before BPE

The GPT-2 tokenizer uses **byte-level BPE**. This means it first converts text into bytes, then learns and applies BPE merges over byte-based symbols.

Text is converted to bytes using an encoding. The most common encoding is **UTF-8**. In UTF-8, simple English letters usually take one byte, while many symbols, emojis, and non-English characters take multiple bytes.

For example:

```text
"Hello 😊" -> UTF-8 bytes -> [72, 101, 108, 108, 111, 32, 240, 159, 152, 138]
```

The smiley emoji is not one byte. It becomes four bytes:

```text
😊 -> [240, 159, 152, 138]
```

### Why there are exactly 256 byte values

A byte is 8 bits, and each bit is either 0 or 1. So a byte has $2^8 = 256$ possible bit patterns. Read as an unsigned integer, a byte is a number from `00000000` = 0 to `11111111` = 255. That is why every value in the lists above lies between 0 and 255:

```text
'H'  -> 72  = 01001000
' '  -> 32  = 00100000
'😊' -> 240, 159, 152, 138 = 11110000 10011111 10011000 10001010
```

Compare this with a character-level base vocabulary. Unicode defines more than 150,000 characters, so a tokenizer that starts from characters can only contain the ones it saw during training. Every other character becomes `[UNK]`. UTF-8, by contrast, writes **every** character as a sequence of 1 to 4 bytes, and every byte is one of only 256 values. A tokenizer whose base vocabulary contains all 256 byte values can therefore represent any text at all, with no unknown tokens.

### How the GPT-2 tokenizer uses the 256 byte values

1. **Base vocabulary.** GPT-2 starts with one token for each of the 256 possible byte values, which occupy token IDs 0 to 255. These IDs are not ordered by byte value; for example, ID 0 is `!` (byte 33). All 256 are included, even though valid UTF-8 text never contains a few of them (192, 193, and 245 to 255), so that any byte sequence has a representation.
2. **Learned merges.** BPE training then runs exactly as in the earlier sections, except that the starting units are bytes instead of characters. GPT-2 learned 50,000 merges, so frequent byte sequences such as `" the"` became single tokens.
3. **Special token.** One extra token, `<|endoftext|>`, marks document boundaries.

So the GPT-2 vocabulary size is $256 + 50{,}000 + 1 = 50{,}257$.

**Bytes are stored as printable symbols.** Some bytes, such as the space (32) and the newline (10), are not printable characters. To store the vocabulary and merge rules as readable text, GPT-2 maps each of the 256 bytes to a visible Unicode character. Printable bytes like `H` keep their own character. The others are shifted to new characters, and that is how the space byte (32) comes to be displayed as `Ġ` (Unicode U+0120, which is 256 + 32). Section 8 uses this marker.

In [7]:
text = "Hello 😊"

# Convert the Python string into UTF-8 bytes.
byte_values = text.encode("utf-8")

print("Original text:", text)
print("Bytes object:", byte_values)
print("Integer byte values:", list(byte_values))

print("\nCharacter-by-character UTF-8 bytes:")
for char in text:
    print(f"{char!r:6s} -> {list(char.encode('utf-8'))}")

Original text: Hello 😊
Bytes object: b'Hello \xf0\x9f\x98\x8a'
Integer byte values: [72, 101, 108, 108, 111, 32, 240, 159, 152, 138]

Character-by-character UTF-8 bytes:
'H'    -> [72]
'e'    -> [101]
'l'    -> [108]
'l'    -> [108]
'o'    -> [111]
' '    -> [32]
'😊'    -> [240, 159, 152, 138]


## 8. Handling Spaces: Why GPT-Style Tokens Often Look Strange

Many BPE tokenizers do not simply split text with Python's `text.split()`. They use a pre-tokenization step that preserves information about spaces.

GPT-2 and RoBERTa-style tokenizers often display a leading-space marker, commonly shown as `Ġ`. For example:

```text
" token" -> "Ġtoken"
```

This means the token includes the information that a space appeared before `token`.

The marker is useful because `token` at the beginning of a sentence and ` token` after a space are not exactly the same text pattern.

In the code below, we will use `Ġ` as a visible marker for a leading space, following the common GPT-2/RoBERTa display convention.

In [8]:
# GPT-2/RoBERTa-style tokenizers often display a leading space as Ġ.
# Here we use the same marker to make spaces visible in the token strings.
SPACE_MARKER = "Ġ"

def simple_pre_tokenize(text):
    """A tiny pre-tokenizer that keeps spaces as a prefix marker."""
    # pieces will store the rough text chunks produced before BPE merges are learned.
    # Example output: ["This", "Ġis", "Ġtokenization", "."]
    pieces = []

    # current accumulates the characters of the word-like piece we are currently reading.
    current = ""

    # This flag remembers that we just saw a space.
    # The next word-like piece should then receive the leading-space marker Ġ.
    at_word_start_after_space = False

    # Read the input text one character at a time.
    for char in text:
        # Case 1: the character is a space, tab, or newline.
        if char.isspace():
            # If we were building a word, finish it before handling the space.
            # Example: after reading "This " we append "This" to pieces.
            if current:
                pieces.append(current)
                current = ""

            # Do not append the actual space as a separate piece.
            # Instead, remember it so the next word can start with Ġ.
            at_word_start_after_space = True

        # Case 2: the character is punctuation that we want as its own piece.
        elif char in ".,!?;:":
            # If punctuation appears after a word, finish the word first.
            # Example: "tokenization." becomes "tokenization" and ".".
            if current:
                pieces.append(current)
                current = ""

            # Keep punctuation as a separate pre-tokenized piece.
            pieces.append(char)

            # After punctuation, the next word should not automatically get Ġ.
            # It will get Ġ only if an actual space is seen next.
            at_word_start_after_space = False

        # Case 3: the character is part of a word-like piece.
        else:
            # If this is the first character after a space, attach the space marker.
            # Example: after reading the space in "This is", "i" starts a new piece "Ġi".
            if not current and at_word_start_after_space:
                current = SPACE_MARKER + char
                at_word_start_after_space = False
            else:
                # Otherwise, continue building the current piece normally.
                current += char

    # If the text ended while we were still building a piece, save that final piece.
    if current:
        pieces.append(current)

    # Return the rough pieces. BPE will later split and merge inside these pieces.
    return pieces

example = "This is tokenization. This is useful!"
print(simple_pre_tokenize(example))

['This', 'Ġis', 'Ġtokenization', '.', 'ĠThis', 'Ġis', 'Ġuseful', '!']


## 9. Training BPE on a Small Sentence Corpus

Now we will train BPE on a small sentence corpus.

The corpus is intentionally tiny so that we can inspect the learned rules. A production tokenizer would be trained on a much larger and more diverse corpus.

In [9]:
sentence_corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will understand how tokenizers generate tokens.",
]

sentence_word_freqs = Counter()
for text in sentence_corpus:
    sentence_word_freqs.update(simple_pre_tokenize(text))

print("Pre-tokenized word frequencies:")
for word, freq in sentence_word_freqs.items():
    print(f"  {word:18s} {freq}")

Pre-tokenized word frequencies:
  This               3
  Ġis                2
  Ġthe               1
  ĠHugging           1
  ĠFace              1
  ĠCourse            1
  .                  4
  Ġchapter           1
  Ġabout             1
  Ġtokenization      1
  Ġsection           1
  Ġshows             1
  Ġseveral           1
  Ġtokenizer         1
  Ġalgorithms        1
  Hopefully          1
  ,                  1
  Ġyou               1
  Ġwill              1
  Ġunderstand        1
  Ġhow               1
  Ġtokenizers        1
  Ġgenerate          1
  Ġtokens            1


In [10]:
sentence_vocab, sentence_merges, sentence_final_splits = train_bpe(sentence_word_freqs, num_merges=20)

print("\nVocabulary size:", len(sentence_vocab))
print("First 20 vocabulary items:", sentence_vocab[:20])
print("Last 10 learned tokens:", sentence_vocab[-10:])

Step 1: merge ('e', 'r') -> 'er'  frequency=6
Step 2: merge ('i', 's') -> 'is'  frequency=5
Step 3: merge ('Ġ', 't') -> 'Ġt'  frequency=5
Step 4: merge ('e', 'n') -> 'en'  frequency=5
Step 5: merge ('Ġt', 'o') -> 'Ġto'  frequency=4
Step 6: merge ('Ġto', 'k') -> 'Ġtok'  frequency=4
Step 7: merge ('Ġtok', 'en') -> 'Ġtoken'  frequency=4
Step 8: merge ('T', 'h') -> 'Th'  frequency=3
Step 9: merge ('Th', 'is') -> 'This'  frequency=3
Step 10: merge ('o', 'u') -> 'ou'  frequency=3
Step 11: merge ('s', 'e') -> 'se'  frequency=3
Step 12: merge ('Ġtoken', 'i') -> 'Ġtokeni'  frequency=3
Step 13: merge ('Ġtokeni', 'z') -> 'Ġtokeniz'  frequency=3
Step 14: merge ('Ġ', 'is') -> 'Ġis'  frequency=2
Step 15: merge ('Ġ', 'a') -> 'Ġa'  frequency=2
Step 16: merge ('a', 't') -> 'at'  frequency=2
Step 17: merge ('i', 'o') -> 'io'  frequency=2
Step 18: merge ('io', 'n') -> 'ion'  frequency=2
Step 19: merge ('Ġ', 'se') -> 'Ġse'  frequency=2
Step 20: merge ('h', 'o') -> 'ho'  frequency=2

Vocabulary size: 50
Fi

### Reading the Merge Rules

Each learned merge says that a pair was common enough to become a single token.

For example, if the algorithm learns:

```text
("i", "s") -> "is"
```

then `is` can become one token. If it later learns:

```text
("Ġ", "is") -> "Ġis"
```

then the word `is` with a leading space can become one token.

This is why GPT-style vocabularies often contain tokens that include a leading-space pattern.

## 10. Tokenizing a New Sentence

To tokenize a full sentence, we:

1. Pre-tokenize it into rough word and punctuation pieces.
2. Split each piece into characters.
3. Apply the learned BPE merges to each piece.
4. Flatten the result into one token sequence.

In [11]:
def tokenize_text(text, merges, base_vocab):
    pieces = simple_pre_tokenize(text)
    tokens = []

    for piece in pieces:
        tokens.extend(tokenize_word(piece, merges, vocab=base_vocab))

    return tokens

sentence_base_vocab = sorted({char for word in sentence_word_freqs for char in word})

test_sentences = [
    "This is tokenization.",
    "This tokenizer generates tokens.",
    "Unknown emojis are hard 😊",
]

for text in test_sentences:
    print(text)
    print(tokenize_text(text, sentence_merges, sentence_base_vocab))
    print()

This is tokenization.
['This', 'Ġis', 'Ġtokeniz', 'at', 'ion', '.']

This tokenizer generates tokens.
['This', 'Ġtokeniz', 'er', 'Ġ', 'g', 'en', 'er', 'at', 'e', 's', 'Ġtoken', 's', '.']

Unknown emojis are hard 😊
['[UNK]', 'n', 'k', 'n', 'o', 'w', 'n', 'Ġ', 'e', 'm', 'o', '[UNK]', 'is', 'Ġa', 'r', 'e', 'Ġ', 'h', 'a', 'r', 'd', 'Ġ', '[UNK]']



The emoji example shows the limitation of this classroom implementation. Since the emoji was never part of the base vocabulary, it becomes `[UNK]`.

A byte-level BPE tokenizer would first convert text to bytes, so even rare characters and emojis could be represented without an unknown token.

## 11. BPE Vocabulary Growth

BPE grows the vocabulary one merge at a time.

Suppose the base vocabulary has 30 symbols and we learn 20 merges. The final vocabulary size is:

```text
30 + 20 = 50
```

In practice, vocabulary size is a design choice. A larger vocabulary can make sequences shorter, but it also makes the embedding table and output layer larger.

| Smaller vocabulary | Larger vocabulary |
| --- | --- |
| Longer token sequences | Shorter token sequences |
| Smaller embedding table | Larger embedding table |
| More splitting into small pieces | More whole words and large subwords |
| Better coverage with fewer tokens | More parameters devoted to token representations |

In [12]:
base_size = len(sentence_base_vocab)
merge_count = len(sentence_merges)
final_size = len(sentence_vocab)

print("Base vocabulary size:", base_size)
print("Number of learned merges:", merge_count)
print("Final vocabulary size:", final_size)
print("Check: base_size + merge_count =", base_size + merge_count)

Base vocabulary size: 30
Number of learned merges: 20
Final vocabulary size: 50
Check: base_size + merge_count = 50


## 12. Modern Byte-Level BPE at Scale

The tokenizers behind current large language models still use the algorithm from this notebook: byte-level BPE, trained by repeatedly merging the most frequent adjacent pair. Two design choices have changed since GPT-2:

1. **Vocabularies are much larger.**
2. **The pre-tokenizer rules are finer.**

| Tokenizer | Used by | Vocabulary size |
| --- | --- | --- |
| `gpt2` | GPT-2 | 50,257 (256 bytes + 50,000 merges + 1 special token) |
| `cl100k_base` | GPT-3.5 Turbo, GPT-4 | about 100,000 |
| `o200k_base` | GPT-4o | about 200,000 |
| Llama 3 | Meta Llama 3 | about 128,000 (100,000 tokens from OpenAI's `tiktoken` + 28,000 added for non-English text) |

### What the pre-tokenizer changes do

Recall from Section 3 that BPE can only merge inside a pre-tokenized chunk. Changing the pre-tokenizer therefore changes which tokens can ever be learned. The `cl100k_base` and `o200k_base` pre-tokenizers differ from GPT-2's in several ways:

- **Numbers are split into chunks of at most three digits.** GPT-2 has learned tokens for some arbitrary digit strings, so similar numbers can split inconsistently, e.g. `1234567 -> 123 | 45 | 67`. With a three-digit limit, the split is always regular: `1234567 -> 123 | 456 | 7`.
- **Runs of spaces become one token.** GPT-2 spends one token per space of indentation. The newer tokenizers merge a run of spaces into a single token, so Python code is much shorter.
- **`o200k_base` also splits at letter-case changes.** Its pre-tokenizer turns `camelCaseVariable` into the chunks `camel | Case | Variable`, while `cl100k_base` keeps it as one chunk for BPE to split.

### Token counts for the same text

The table below was recorded with `tiktoken` 0.9.0. It shows how many tokens each tokenizer needs for the same input:

| Input | `gpt2` | `cl100k_base` | `o200k_base` |
| --- | --- | --- | --- |
| `Tokenization is useful.` | 5 | 5 | 5 |
| `The year 2024 had 365 days and 1234567 seconds.` | 12 | 16 | 16 |
| A two-line Python loop indented with spaces | 22 | 12 | 12 |
| `नमस्ते, आप कैसे हैं?` (Hindi: "Hello, how are you?") | 32 | 20 | 9 |

How to read these results:

- **Ordinary English is already efficient.** All three tokenizers need 5 tokens for the first sentence.
- **Shorter is not always the goal.** The number sentence uses *more* tokens in the newer tokenizers, because every space before a number becomes its own token and digits are capped at three per token. The model gets a consistent view of numbers in exchange.
- **Indentation is cheap in the newer tokenizers.** Grouping runs of spaces nearly halves the token count for the Python snippet.
- **The training corpus decides who pays.** GPT-2's merges were learned mostly from English text. A Hindi sentence therefore falls back to raw UTF-8 bytes, which is 3 bytes per Devanagari character, as Section 7 explained. With a larger and more multilingual vocabulary, `o200k_base` needs 9 tokens instead of 32. More tokens per sentence means higher API cost, less text fitting in the context window, and often lower quality for that language.

References:
- OpenAI `tiktoken` library: https://github.com/openai/tiktoken
- Llama Team, AI @ Meta (2024), "The Llama 3 Herd of Models", arXiv:2407.21783 (tokenizer description).

### What is `tiktoken`?

`tiktoken` is OpenAI's open-source library for byte-level BPE tokenization (MIT license). It contains the exact tokenizers that OpenAI's models use, so you can encode, decode, and count tokens locally the same way the model sees them:

```python
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
ids = enc.encode("Tokenization is useful.")  # [3404, 2065, 374, 5505, 13]
enc.decode(ids)                              # 'Tokenization is useful.'
```

Each named encoding, such as `gpt2`, `cl100k_base`, or `o200k_base`, has the same two parts as the tokenizer built in this notebook:

| Part of an encoding | Counterpart in this notebook |
| --- | --- |
| A pre-tokenizer regular expression that splits text into chunks | `simple_pre_tokenize` (Section 8) |
| A merge table: every learned byte sequence with its **rank**, the order in which it was learned | the ordered `merges` list (Section 5) |

To tokenize a chunk, `tiktoken` repeatedly merges the adjacent pair with the lowest rank, meaning the merge learned earliest. This is the Section 6 rule, "apply merges in the order they were learned", implemented efficiently in Rust instead of in a Python loop.

Two practical notes:

- **It only applies tokenizers; it does not train them.** To learn merges from your own corpus, use a library such as Hugging Face `tokenizers` or SentencePiece.
- **It is commonly used to count tokens** before calling an API, in order to estimate cost and stay within a model's context window. `tiktoken.encoding_for_model("gpt-4o")` returns the matching encoding, here `o200k_base`.

The next cell uses `tiktoken` to reproduce the token-count table above. It is **optional and off by default**: it needs the `tiktoken` package and downloads each tokenizer's merge file (about 1 to 4 MB each) the first time it runs. Set `RUN_TIKTOKEN_DEMO = True` to run it.

In [ ]:
# Optional: needs the tiktoken package and downloads each tokenizer's merge file
# (about 1 to 4 MB each) on first use. Files are cached by tiktoken afterwards.
RUN_TIKTOKEN_DEMO = False

samples = {
    "English": "Tokenization is useful.",
    "Numbers": "The year 2024 had 365 days and 1234567 seconds.",
    "Python code": "    for i in range(10):\n        print(i)",
    "Hindi": "नमस्ते, आप कैसे हैं?",
}
encoding_names = ["gpt2", "cl100k_base", "o200k_base"]

if not RUN_TIKTOKEN_DEMO:
    print("Skipped. Set RUN_TIKTOKEN_DEMO = True to download the tokenizers and compare them.")
else:
    try:
        import tiktoken
    except ImportError:
        tiktoken = None
        print("tiktoken is not installed. Install it with: pip install tiktoken")

    if tiktoken is not None:
        encodings = {name: tiktoken.get_encoding(name) for name in encoding_names}

        print(f"{'Input':12s}" + "".join(f"{name:>14s}" for name in encoding_names))
        for label, text in samples.items():
            counts = [len(encodings[name].encode(text)) for name in encoding_names]
            print(f"{label:12s}" + "".join(f"{count:14d}" for count in counts))

        # Show the actual pieces for the number sentence. Each token is a sequence of bytes;
        # decode_single_token_bytes returns those bytes, which we decode for display.
        print("\nNumber sentence, token by token:")
        for name in encoding_names:
            enc = encodings[name]
            pieces = [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace")
                      for t in enc.encode(samples["Numbers"])]
            print(f"  {name:12s} {pieces}")

## 13. BPE vs WordPiece

BPE and WordPiece are both subword tokenization algorithms, but their training choices differ.

| Algorithm | Used by | Main training idea |
| --- | --- | --- |
| BPE | GPT, GPT-2, RoBERTa, BART | Repeatedly merge the most frequent adjacent pair |
| WordPiece | BERT | Choose merges using a likelihood-based score rather than raw pair frequency |

The high-level goal is similar: build a vocabulary of useful subword units so the model can handle common words efficiently and still represent rare words through smaller pieces.

## 14. Classroom Exercise

Try answering these before running more experiments.

1. In the toy corpus, why does `("u", "g")` merge before `("h", "u")`?
2. If a pair appears in a very frequent word, how does that affect its chance of being merged?
3. Why does BPE need to apply merge rules in the same order in which they were learned?
4. What problem does byte-level BPE solve compared with character-level BPE?
5. What is the trade-off between a small vocabulary and a large vocabulary?

## 15. Key Takeaways

- BPE is a **data-driven subword tokenization algorithm**.
- It starts with small units, such as characters or bytes.
- During training, it repeatedly merges the most frequent adjacent token pair.
- The learned merge rules define how new text is tokenized.
- Frequent words may become single tokens; rare words are usually broken into smaller subwords.
- Byte-level BPE starts from bytes, so it can represent almost any text without needing an unknown character token.
- Modern LLM tokenizers use the same byte-level BPE algorithm with much larger vocabularies and finer pre-tokenization rules, which change token counts for numbers, code, and non-English text.
- BPE affects model efficiency because tokenization determines sequence length, vocabulary size, embedding size, and output-layer size.